# Projet Trafic aérien — ADP
Vols au départ de New York (JFK, LGA, EWR), source BTS.

⚠️ Limites : le fichier `flights` ne contient pas les vols du 4 juillet au 30 septembre ; les vols sont datés 2021 mais ce sont ceux de 2013.

## Chargement des données

In [ ]:
import pandas as pd
df1= pd.read_excel("/airports.xlsx")
df1

In [ ]:
import pandas as pd

df2 = pd.read_excel("/flights.xlsx", header=None)[0].str.split(",", expand=True)
df2.columns = df2.iloc[0]
df2 = df2[1:].reset_index(drop=True)

# Les valeurs manquantes sont codées par un espace " " → vraies cases vides
df2 = df2.replace(" ", None)
df2

# MISSION 1 : Se familiariser avec les données

QST 1:
    Combien y-a-t-il :  

    d’aéroports en tout, de départ et de destination
    Combien y-a-t-il d’aéroports où on ne passe pas à l’heure d’été (indice : colonne dst : 23) ? de fuseaux horaires (10 voir colone tzone dont une  « N ») de compagnies, d’avions, de vols annulés ?  

In [ ]:
print("Aéroports en tout :", df1["faa"].nunique())
print("Aéroports de départ :", df2["origin"].nunique(), list(df2["origin"].unique()))
print("Aéroports de destination :", df2["dest"].nunique())

In [ ]:
print("Aéroports sans heure d'été (dst = N) :", (df1["dst"] == "N").sum())
print("Fuseaux horaires :", df1["tzone"].nunique())

Parmi les 10 fuseaux, `\N` (3 aéroports) est une valeur manquante : il y a donc 9 vrais fuseaux horaires.

In [ ]:
# Compagnies : nombre de codes compagnie différents
print("Compagnies :", df2["carrier"].nunique())

# Avions : nombre d'immatriculations différentes
print("Avions :", df2["tailnum"].nunique())

# Vols annulés : vols sans heure de départ réelle
print("Vols annulés :", df2["dep_time"].isna().sum())

2. Quel est l’aéroport de départ le plus emprunté ? Quelles sont les 10 destinations les plus (moins)
prisées (en indiquant le nom complet des destinations et non juste le code correspondant et le %
corresp pour chaque destination) ? Quelle sont les 10 avions qui ont le plus (moins) décollé ?

In [ ]:
print(df2["origin"].value_counts())

In [ ]:
# Compter les vols par destination (table flights = df2)
dest = df2["dest"].value_counts().reset_index()
dest.columns = ["faa", "nb_vols"]

# Calculer le %
dest["pourcentage"] = (dest["nb_vols"] / len(df2) * 100).round(2)

# Ajouter le nom complet (table airports = df1)
dest = dest.merge(df1[["faa", "name"]], on="faa", how="left")
print(dest)
print("10 destinations les PLUS prisées :")
print(dest.head(10))

print("10 destinations les MOINS prisées :")
print(dest.tail(10))

In [ ]:
# On garde seulement les vols qui ont vraiment décollé (dep_time non vide)
decolles = df2[df2["dep_time"].notna() & df2["tailnum"].notna()]
avions = decolles["tailnum"].value_counts()
print("10 avions qui ont le PLUS décollé :")
print(avions.head(10))

print("10 avions qui ont le MOINS décollé :")
print(avions.tail(10))

3. Combien chaque compagnie a desservi de destination ; combien chaque compagnie a desservie de
destination par aéroport d’origine. Réaliser les graphiques adéquats qui synthétisent ces
informations ?

In [ ]:
import matplotlib.pyplot as plt

# a) Nombre de destinations par compagnie
a = df2.groupby("carrier")["dest"].nunique()
print(a)
a.plot(kind="bar", title="Destinations par compagnie")
plt.show()

# b) Nombre de destinations par compagnie et par aéroport d'origine
b = df2.groupby(["carrier", "origin"])["dest"].nunique().unstack(fill_value=0)
print(b)
b.plot(kind="bar", title="Destinations par compagnie et par aéroport")
plt.show()

4. Trouver tous les vols ayant atterri à Houston (IAH ou HOU) ? Combien de vols partent de NYC
airports vers Seattle, combien de compagnies desservent cette destination et combien d’avions
“uniques” ?

In [ ]:
# Vols ayant atterri = vers IAH ou HOU, avec une heure d'arrivée réelle
houston = df2[df2["dest"].isin(["IAH", "HOU"]) & df2["arr_time"].notna()]
print("Vols atterris à Houston :", len(houston))
print(houston["dest"].value_counts())

In [ ]:
# 1. Garder les vols qui partent de NYC (EWR, JFK, LGA) et vont à Seattle (SEA)
seattle = df2[(df2["origin"].isin(["EWR", "JFK", "LGA"])) & (df2["dest"] == "SEA")]

# 2. Nombre de vols
print("Vols NYC → Seattle :", len(seattle))

# 3. Nombre de compagnies
print("Compagnies :", seattle["carrier"].nunique())

# 4. Nombre d'avions uniques
print("Avions uniques :", seattle["tailnum"].nunique())

5. Trouver le nombre de vols par destination ? Trier les vols suivant la destination, l’aéroport d’origine,
la compagnie dans un ordre alphabétique croissant (en réalisant les jointures nécessaires pour
obtenir les noms explicites des aéroports) ?

In [ ]:
# 1. Compter les vols par destination
df2["dest"].value_counts()

In [ ]:
noms = df1.set_index("faa")["name"]
df2["nom_dest"] = df2["dest"].map(noms)
df2["nom_origine"] = df2["origin"].map(noms)
df2 = df2.sort_values(["nom_dest", "nom_origine", "carrier"])

df2[["nom_dest", "nom_origine", "carrier"]].head(20)

6. Quelles sont les compagnies qui n'opèrent pas sur tous les aéroports d’origine ? Quelles sont les
compagnies qui desservent l’ensemble de destinations ?
Faire un tableau où l’on récupère l’ensemble des origines et des destinations pour l’ensemble des
compagnies.

In [ ]:
# a) Compagnies qui n'opèrent pas sur les 3 aéroports d'origine
origines = df2.groupby("carrier")["origin"].nunique()
print("Compagnies qui n'opèrent pas sur tous les aéroports :")
print(origines[origines < 3])

# b) Compagnies qui desservent toutes les destinations
destinations = df2.groupby("carrier")["dest"].nunique()
print("Compagnies qui desservent toutes les destinations :")
print(destinations[destinations == df2["dest"].nunique()])

# c) Tableau des origines et destinations de chaque compagnie
tableau = df2[["carrier", "origin", "dest"]].drop_duplicates()
tableau

7. Quelles sont les destinations qui sont exclusives à certaines compagnies ?

In [ ]:
# Nombre de compagnies différentes pour chaque destination
compagnies = df2.groupby("dest")["carrier"].nunique()

# Destinations desservies par une seule compagnie
exclusives = compagnies[compagnies == 1]
print("Nombre de destinations exclusives :", len(exclusives))

# Afficher la destination et sa compagnie
df2[df2["dest"].isin(exclusives.index)][["dest", "carrier"]].drop_duplicates()

8. Filtrer le vol pour trouver ceux exploités par United, American ou Delta ?

In [ ]:
# Garder les vols de United (UA), American (AA) ou Delta (DL)
vld = df2[df2["carrier"].isin(["UA", "AA", "DL"])]

print("Nombre de vols :", len(vld))
print(vld["carrier"].value_counts())

vld

# MISSION 2 : Créer la base de données (PostgreSQL / Supabase)
Les scripts SQL exécutés dans le SQL Editor de Supabase sont dans le dossier `sql/` du dépôt :
1. `01_creation_tables.sql` : tables, clés primaires et étrangères, contrôles regex ;
2. `02_controles_qualite.sql` : vérification des regex, PK et FK sur les données brutes ;
3. `03_nettoyage_insertion.sql` : correction des anomalies et insertion ;
4. `04_optimisation_types.sql` : SMALLINT / REAL (−23 % de stockage) ;
5. `05_privileges.sql` : groupes lecture / écriture et comptes des membres.

### Connexion sécurisée
Mot de passe saisi avec `getpass` (jamais écrit dans le code), connexion chiffrée (`sslmode=require`).

In [ ]:
!pip install sqlalchemy psycopg2-binary nycflights13

from getpass import getpass
from sqlalchemy import create_engine, URL

mot_de_passe = getpass("Mot de passe Supabase : ")

url = URL.create(
    "postgresql+psycopg2",
    username="postgres.fozjmzskpszdpjeriywn",
    password=mot_de_passe,
    host="aws-0-eu-west-1.pooler.supabase.com",
    port=5432,
    database="postgres",
    query={"sslmode": "require"},
)
engine = create_engine(url)

with engine.connect():
    print("✅ Connecté à Supabase !")

### Envoi des données brutes (exécuté une seule fois, avant les scripts SQL 02 et 03)

In [ ]:
from nycflights13 import airlines, planes, weather

airports_brut = df1.drop(columns="Unnamed: 0")      # colonne de numéros inutile
flights_brut = df2[df2.columns[:19]]                # les 19 colonnes d'origine

for nom, table in [("brut_airlines", airlines), ("brut_airports", airports_brut), ("brut_planes", planes),
                   ("brut_weather", weather), ("brut_flights", flights_brut)]:
    table.to_sql(nom, engine, if_exists="replace", index=False, method="multi", chunksize=5000)
    print(nom, "envoyée :", len(table), "lignes")

### Vérification avec Python après nettoyage et insertion

In [ ]:
import pandas as pd

# Lire les tables depuis Supabase
flights  = pd.read_sql("SELECT * FROM flights", engine)
airports = pd.read_sql("SELECT * FROM airports", engine)
planes   = pd.read_sql("SELECT * FROM planes", engine)
weather  = pd.read_sql("SELECT * FROM weather", engine)

# 1. Clés primaires
print("Doublons 'flight' seul :", flights["flight"].duplicated().sum())
print("Doublons combinaison flights :", flights.duplicated(["year","month","day","hour","carrier","flight"]).sum())
print("Doublons weather :", weather.duplicated(["year","month","day","hour","origin"]).sum())

# 2. Clés étrangères
print("dest absents de airports :", set(flights["dest"]) - set(airports["faa"]))
print("tailnum absents de planes :", set(flights["tailnum"].dropna()) - set(planes["tailnum"]))

# 3. Regex
print("tailnum invalides :", (~flights["tailnum"].dropna().str.fullmatch(r"N[A-Z0-9]{1,5}")).sum())

# REPORTING
## 3.1 Pics de trafic
### Q1 : transformer les heures en vraies dates

In [ ]:
import pandas as pd

flights = pd.read_sql("SELECT * FROM flights", engine)
flights[["year", "month", "day", "hour", "minute", "sched_dep_time", "dep_time"]].head()

In [ ]:
# 515 = 5 h 15 → hour × 100 + minute = 5 × 100 + 15 = 515
print((flights["hour"] * 100 + flights["minute"] == flights["sched_dep_time"]).all())

In [ ]:
vols = flights.copy()

# 1. Départ prévu : on assemble year, month, day, hour, minute en une date
vols["sched_dep_time"] = pd.to_datetime(vols[["year", "month", "day", "hour", "minute"]])

# 2. Départ réel = départ prévu + retard au départ (en minutes)
vols["dep_time"] = vols["sched_dep_time"] + pd.to_timedelta(vols["dep_delay"], unit="min")

# 3. Arrivée prévue : la date du vol + l'heure (format HHMM)
date = pd.to_datetime(vols[["year", "month", "day"]])
heure = vols["sched_arr_time"] // 100
minute = vols["sched_arr_time"] % 100
vols["sched_arr_time"] = date + pd.to_timedelta(heure * 60 + minute, unit="min")

#    si l'arrivée prévue est avant le départ, l'avion arrive le lendemain
vols.loc[vols["sched_arr_time"] < vols["sched_dep_time"], "sched_arr_time"] += pd.Timedelta(days=1)

# 4. Arrivée réelle = arrivée prévue + retard à l'arrivée
vols["arr_time"] = vols["sched_arr_time"] + pd.to_timedelta(vols["arr_delay"], unit="min")

# 5. Supprimer les colonnes devenues inutiles
vols = vols.drop(columns=["year", "month", "day", "hour", "minute"])

vols[["sched_dep_time", "dep_time", "sched_arr_time", "arr_time"]].head()

### Q2 : trafic mensuel des 3 aéroports (en facettes)

In [ ]:
import matplotlib.pyplot as plt

# 1. Extraire le mois et le jour de la date de départ prévue
vols["mois"] = vols["sched_dep_time"].dt.month
vols["jour"] = vols["sched_dep_time"].dt.date

# 2. Pour chaque aéroport et chaque mois : nombre de vols et nombre de jours
trafic = (vols.groupby(["origin", "mois"])
              .agg(nb_vols=("flight", "size"), nb_jours=("jour", "nunique"))
              .reset_index())

# 3. L'indicateur : nombre moyen de vols par jour
trafic["vols_par_jour"] = trafic["nb_vols"] / trafic["nb_jours"]

# 4. On retire les mois incomplets (juillet n'a que 3 jours dans nos données)
trafic = trafic[trafic["nb_jours"] >= 28]

# 5. Le graphique : une facette par aéroport + la ligne de moyenne
noms_mois = ["Jan", "Fév", "Mar", "Avr", "Mai", "Juin", "Juil", "Août", "Sept", "Oct", "Nov", "Déc"]
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True, sharey=True)

for ax, aeroport in zip(axes, ["EWR", "JFK", "LGA"]):
    t = trafic[trafic["origin"] == aeroport]
    moyenne = t["vols_par_jour"].mean()
    ax.bar(t["mois"], t["vols_par_jour"], color="steelblue")
    ax.axhline(moyenne, color="red", linestyle="--", label=f"Moyenne : {moyenne:.0f} vols/jour")
    ax.set_title(aeroport)
    ax.set_ylabel("Vols par jour")
    ax.legend(loc="lower right")

axes[-1].set_xticks(range(1, 13), noms_mois)
fig.suptitle("Trafic mensuel moyen par aéroport d'origine")
plt.tight_layout()
plt.savefig("trafic_mensuel.png", dpi=150)
plt.show()

### Q3 : taux d'accroissement mensuel

In [ ]:
# 1. Taux d'accroissement par rapport au mois précédent, pour chaque aéroport
trafic["taux"] = trafic.groupby("origin")["vols_par_jour"].pct_change() * 100

# 2. Pas de taux quand le mois précédent manque (octobre comparé à juin = faux)
mois_precedent = trafic.groupby("origin")["mois"].shift(1)
trafic.loc[trafic["mois"] - mois_precedent != 1, "taux"] = None

print(trafic[["origin", "mois", "vols_par_jour", "taux"]].round(1))

# 3. Le graphique : à gauche le trafic, à droite le taux d'accroissement
fig, axes = plt.subplots(3, 2, figsize=(14, 9), sharex=True)

for i, aeroport in enumerate(["EWR", "JFK", "LGA"]):
    t = trafic[trafic["origin"] == aeroport]
    moyenne = t["vols_par_jour"].mean()

    # Colonne de gauche : trafic + moyenne
    ax = axes[i, 0]
    ax.bar(t["mois"], t["vols_par_jour"], color="steelblue")
    ax.axhline(moyenne, color="red", linestyle="--", label=f"Moyenne : {moyenne:.0f} vols/jour")
    ax.set_title(f"{aeroport} : vols par jour")
    ax.set_ylim(0, 380)
    ax.legend(loc="lower right")

    # Colonne de droite : taux d'accroissement
    ax = axes[i, 1]
    ax.plot(t["mois"], t["taux"], marker="o", color="darkorange")
    ax.axhline(0, color="grey", linewidth=1)
    ax.set_title(f"{aeroport} : taux d'accroissement mensuel (%)")
    ax.set_ylim(-7, 7)

for ax in axes[-1]:
    ax.set_xticks(range(1, 13), noms_mois)

fig.suptitle("Trafic mensuel et taux d'accroissement par aéroport d'origine")
plt.tight_layout()
plt.savefig("trafic_et_accroissement.png", dpi=150)
plt.show()

### Q4 : top 3 des aéroports d'origine et de destination

In [ ]:
airports = pd.read_sql("SELECT faa, name FROM airports", engine)
noms = airports.set_index("faa")["name"]

# Top 3 des aéroports d'origine
top_origine = vols["origin"].value_counts().head(3).reset_index()
top_origine.columns = ["code", "nb_vols"]
top_origine["nom"] = top_origine["code"].map(noms)
top_origine["%"] = (top_origine["nb_vols"] / len(vols) * 100).round(1)
print("Top 3 des aéroports d'ORIGINE :")
print(top_origine)

# Top 3 des aéroports de destination
top_dest = vols["dest"].value_counts().head(3).reset_index()
top_dest.columns = ["code", "nb_vols"]
top_dest["nom"] = top_dest["code"].map(noms)
top_dest["%"] = (top_dest["nb_vols"] / len(vols) * 100).round(1)
print("Top 3 des aéroports de DESTINATION :")
print(top_dest)

### Q5 : filtrer les vols

In [ ]:
effectues = vols[vols["dep_time"].notna()].copy()
d = effectues["dep_time"]          # raccourci : la date et l'heure de départ réelle

# a) Le 1er janvier
jan1 = effectues[(d.dt.month == 1) & (d.dt.day == 1)]
print("Vols le 1er janvier :", len(jan1))

#    Combien de vols chaque jour
effectues["date"] = d.dt.date
par_jour = effectues.groupby("date").size()
print(par_jour)
print("Moyenne par jour :", round(par_jour.mean()))

# b) En novembre ou en décembre
nov_dec = effectues[d.dt.month.isin([11, 12])]
print("Vols en novembre-décembre :", len(nov_dec))

# c) Les jours spéciaux
speciaux = effectues[
    ((d.dt.month == 12) & (d.dt.day == 25)) |   # Christmas Day
    ((d.dt.month == 1)  & (d.dt.day == 1))  |   # Jour de l'an
    ((d.dt.month == 7)  & (d.dt.day == 4))  |   # Independence Day
    ((d.dt.month == 11) & (d.dt.day == 29))     # Thanksgiving (date de l'énoncé)
]
print("Vols les jours spéciaux :", len(speciaux))
print(speciaux["dep_time"].dt.strftime("%d/%m").value_counts())

# d) En été (juillet, août, septembre)
ete = effectues[d.dt.month.isin([7, 8, 9])]
print("Vols en été :", len(ete))

# e) Départ entre minuit et 6 h (inclus)
nuit = effectues[(d.dt.hour < 6) | ((d.dt.hour == 6) & (d.dt.minute == 0))]
print("Vols entre 00:00 et 06:00 :", len(nuit))

In [ ]:
jan1 = effectues[d.dt.date == pd.Timestamp("2021-01-01").date()]
print("Vols le 1er janvier 2021 :", len(jan1))   # 837

⚠️ Le 4 juillet et l'été sont quasi vides (fichier arrêté au 3 juillet). Thanksgiving 2013 tombait le **28/11**, pas le 29/11 de l'énoncé.

### Q6 : semaine contre week-end, jours spéciaux contre moyenne
On utilise le vrai calendrier 2013 pour les jours de la semaine.

In [ ]:
# Date du vol avec la vraie année (2013 = 2021 - 8 ans)
vols["date"] = vols["sched_dep_time"].dt.normalize() - pd.DateOffset(years=8)

# Nombre de vols par jour (sans le 3 juillet, dernier jour incomplet du fichier)
par_jour = vols.groupby("date").size().rename("nb_vols").reset_index()
par_jour = par_jour[par_jour["date"] != "2013-07-03"]

# Jour de la semaine : 0 = lundi ... 5 = samedi, 6 = dimanche
par_jour["jour_semaine"] = par_jour["date"].dt.dayofweek
par_jour["type_jour"] = par_jour["jour_semaine"].map(lambda j: "Week-end" if j >= 5 else "Jour ouvré")

# 1. Week-end ou jours ouvrés ?
print(par_jour.groupby("type_jour")["nb_vols"].mean().round(0))

# 2. Jours spéciaux comparés à la moyenne annuelle
moyenne = par_jour["nb_vols"].mean()
print("Moyenne annuelle :", round(moyenne), "vols par jour")

speciaux = {"2013-01-01": "Jour de l'an", "2013-11-28": "Thanksgiving",
            "2013-11-29": "29/11 (date de l'énoncé)", "2013-12-25": "Noël"}
for date, nom in speciaux.items():
    n = par_jour.loc[par_jour["date"] == date, "nb_vols"].iloc[0]
    print(f"{nom} : {n} vols ({(n / moyenne - 1) * 100:+.0f} % par rapport à la moyenne)")

In [ ]:
noms_jours = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
speciaux_graph = {"2013-01-01": "Jour de l'an", "2013-11-28": "Thanksgiving", "2013-12-25": "Noël"}

fig, axes = plt.subplots(2, 1, figsize=(12, 9), gridspec_kw={"height_ratios": [2, 1]})

# Graphique 1 : vols par jour sur l'année + moyenne + jours spéciaux
ax = axes[0]
courbe = par_jour.set_index("date")["nb_vols"].asfreq("D")    # jours manquants = trou dans la courbe
ax.plot(courbe.index, courbe.values, color="steelblue", linewidth=1)
ax.axhline(moyenne, color="red", linestyle="--", label=f"Moyenne : {moyenne:.0f} vols/jour")
for date, nom in speciaux_graph.items():
    n = courbe[date]
    ax.scatter(pd.Timestamp(date), n, color="darkorange", zorder=3)
    ax.annotate(f"{nom}\n{n:.0f} vols", (pd.Timestamp(date), n), textcoords="offset points",
                xytext=(0, -32), ha="center", fontsize=9)
ax.set_ylim(500, 1100)
ax.set_title("Nombre de vols par jour (les creux réguliers sont les samedis)")
ax.set_ylabel("Vols par jour")
ax.legend(loc="upper left")

# Graphique 2 : moyenne par jour de la semaine
ax = axes[1]
semaine = par_jour.groupby("jour_semaine")["nb_vols"].mean()
ax.bar(noms_jours, semaine, color=["steelblue"] * 5 + ["darkorange"] * 2)
for i, v in enumerate(semaine):
    ax.text(i, v + 10, f"{v:.0f}", ha="center", fontsize=9)
ax.set_ylim(0, 1100)
ax.set_title("Nombre moyen de vols selon le jour de la semaine (orange = week-end)")
ax.set_ylabel("Vols par jour")

plt.tight_layout()
plt.savefig("trafic_jours.png", dpi=150)
plt.show()

## 3.2 Retards
### Q1 : vols les plus retardés

In [ ]:
colonnes = ["sched_dep_time", "carrier", "flight", "origin", "dest", "dep_delay", "arr_delay"]

# a) Les plus retardés à l'ARRIVÉE
print("Plus retardés à l'arrivée :")
print(vols.sort_values("arr_delay", ascending=False)[colonnes].head(10))

# b) Les plus retardés au DÉPART
print("Plus retardés au départ :")
print(vols.sort_values("dep_delay", ascending=False)[colonnes].head(10))

# c) Au départ ET à l'arrivée à la fois : on additionne les deux retards
vols["retard_total"] = vols["dep_delay"] + vols["arr_delay"]
print("Plus retardés au départ et à l'arrivée :")
print(vols.sort_values("retard_total", ascending=False)[colonnes + ["retard_total"]].head(10))

# Combien de vols sont en retard aux deux moments ?
deux = vols[(vols["dep_delay"] > 0) & (vols["arr_delay"] > 0)]
print("Vols en retard au départ ET à l'arrivée :", len(deux))

### Q2 : retard moyen au départ, global et par jour

In [ ]:
# a) Retard moyen au départ pour l'ensemble des vols
print("Retard moyen au départ :", round(vols["dep_delay"].mean(), 1), "minutes")
print("Retard médian au départ :", vols["dep_delay"].median(), "minutes")

# b) Retard moyen au départ pour chaque jour
retard_jour = vols.groupby("date")["dep_delay"].mean().round(1)
print(retard_jour)

print("Les 5 pires jours :")
print(retard_jour.sort_values(ascending=False).head(5))

# Graphique
courbe = retard_jour.asfreq("D")        # coupe la courbe pendant les mois sans données
plt.figure(figsize=(12, 4))
plt.plot(courbe.index, courbe.values, color="steelblue", linewidth=1)
plt.axhline(vols["dep_delay"].mean(), color="red", linestyle="--", label="Moyenne annuelle")
plt.title("Retard moyen au départ, jour par jour")
plt.ylabel("Minutes")
plt.legend()
plt.tight_layout()
plt.savefig("retard_jour.png", dpi=150)
plt.show()

### Q3 : retards de plus de 2 heures

In [ ]:
# a) Arrivés avec plus de 2 h de retard, alors qu'ils étaient partis à l'heure
q3a = vols[(vols["arr_delay"] > 120) & (vols["dep_delay"] <= 0)]
print("Partis à l'heure mais arrivés avec + de 2 h de retard :", len(q3a))
print(q3a[["sched_dep_time", "carrier", "flight", "origin", "dest", "dep_delay", "arr_delay"]])

# b) Ni plus de 2 h de retard au départ, ni plus de 2 h à l'arrivée
q3b = vols[(vols["dep_delay"] <= 120) & (vols["arr_delay"] <= 120)]
print("Pas plus de 2 h de retard au départ ni à l'arrivée :", len(q3b))

### Q4 : vols partis ou arrivés en avance

In [ ]:
# Décollé plus tôt que prévu
en_avance_dep = vols[vols["dep_delay"] < 0]
print("Décollé plus tôt que prévu :", len(en_avance_dep))

# Atterri plus tôt que prévu
en_avance_arr = vols[vols["arr_delay"] < 0]
print("Atterri plus tôt que prévu :", len(en_avance_arr))

# Les deux à la fois
en_avance_deux = vols[(vols["dep_delay"] < 0) & (vols["arr_delay"] < 0)]
print("En avance au départ ET à l'arrivée :", len(en_avance_deux))

### Q5 et Q6 : retard rattrapé en vol (gain)

In [ ]:
# ============ QUESTION 5 : partis avec 1 h de retard ou plus, rattrapé plus de 30 min ============

# Version 1 : en créant la variable gain
vols["gain"] = vols["arr_delay"] - vols["dep_delay"]
q5 = vols[(vols["dep_delay"] >= 60) & (vols["gain"] < -30)]
print("Q5 avec la variable gain :", len(q5))

# Version 2 : sans créer de variable
q5_bis = vols[(vols["dep_delay"] >= 60) & (vols["dep_delay"] - vols["arr_delay"] > 30)]
print("Q5 sans variable :", len(q5_bis))

# ============ QUESTION 6 : gain par heure de vol ============

# Version 1 : en créant les variables hours et gain_per_hour
rattrapes = vols[(vols["dep_delay"] > 0) & (vols["gain"] < 0)].copy()
rattrapes["hours"] = rattrapes["air_time"] / 60
rattrapes["gain_per_hour"] = rattrapes["gain"] / rattrapes["hours"]
print("\nQ6 : vols qui ont rattrapé du retard :", len(rattrapes))
print("Q6 avec variables :")
print(rattrapes["gain_per_hour"].describe().round(1))

# Version 2 : sans créer de variable
masque = (vols["dep_delay"] > 0) & (vols["arr_delay"] - vols["dep_delay"] < 0)
gain_ph = (vols["arr_delay"] - vols["dep_delay"]) / (vols["air_time"] / 60)
print("\nQ6 sans variable :")
print(gain_ph[masque].describe().round(1))

# Les 5 vols qui ont rattrapé le plus par heure
print("\nRecords de rattrapage :")
print(rattrapes.sort_values("gain_per_hour")[["carrier", "flight", "origin", "dest",
      "dep_delay", "arr_delay", "air_time", "gain", "gain_per_hour"]].head(5).round(1))

### Q7 : vitesse et distance

In [ ]:
colonnes = ["carrier", "flight", "origin", "dest", "distance", "air_time"]

# ============ a) Trier les vols selon leur vitesse ============

# Version 1 : en créant la variable speed (en miles par heure)
vols["speed"] = vols["distance"] / vols["air_time"] * 60
print("Les 5 vols les PLUS RAPIDES :")
print(vols.sort_values("speed", ascending=False)[colonnes + ["speed"]].head(5).round(0))
print("\nLes 5 vols les PLUS LENTS :")
print(vols.sort_values("speed")[colonnes + ["speed"]].head(5).round(0))

# Version 2 : sans créer de variable (on trie avec l'ordre du calcul)
ordre = (vols["distance"] / vols["air_time"] * 60).sort_values(ascending=False).index
print("\nSans variable, les 5 plus rapides :")
print(vols.loc[ordre, colonnes].head(5))

# ============ b) Distance la plus grande / la plus petite ============
print("\nLONG-COURRIER (distance la plus grande) :")
print(vols.sort_values("distance", ascending=False)[colonnes].head(3))
print("\nCOURT-COURRIER (distance la plus petite) :")
print(vols.sort_values("distance")[colonnes].head(3))

### Indicateur de qualité et classement des compagnies

In [ ]:
airlines = pd.read_sql("SELECT * FROM airlines", engine)

# Vols arrivés (on enlève les annulés et les déroutés)
arrives = vols[vols["arr_delay"].notna()].copy()
arrives["en_retard"] = arrives["arr_delay"] > 15

# ============ 1. Qualité par COMPAGNIE ============
qualite = (arrives.groupby("carrier")
                  .agg(nb_vols=("en_retard", "size"), pct_retard=("en_retard", "mean"))
                  .reset_index())
qualite["pct_retard"] = (qualite["pct_retard"] * 100).round(1)
qualite = qualite[qualite["nb_vols"] >= 100]          # on écarte les compagnies avec trop peu de vols
qualite = qualite.merge(airlines, on="carrier").sort_values("pct_retard")
print(qualite)

# ============ 2. Qualité par LIGNE (compagnie + origine + destination) ============
lignes = (arrives.groupby(["carrier", "origin", "dest"])
                 .agg(nb_vols=("en_retard", "size"), pct_retard=("en_retard", "mean"))
                 .reset_index())
lignes = lignes[lignes["nb_vols"] >= 100]
lignes["pct_retard"] = (lignes["pct_retard"] * 100).round(1)
print("\nLes 5 lignes les MOINS fiables :")
print(lignes.sort_values("pct_retard", ascending=False).head(5))
print("\nLes 5 lignes les PLUS fiables :")
print(lignes.sort_values("pct_retard").head(5))

# ============ 3. Graphique : classement des compagnies ============
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(qualite["name"], qualite["pct_retard"], color="steelblue")
for i, p in enumerate(qualite["pct_retard"]):
    ax.text(p + 0.5, i, f"{p} %", va="center", fontsize=9)
ax.set_xlabel("% de vols arrivés avec plus de 15 min de retard")
ax.set_title("Classement des compagnies par fiabilité (la plus fiable en haut)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("classement_compagnies.png", dpi=150)
plt.show()

## 3.3 Le retard augmente-t-il avec la distance ?
### Q1 à Q5

In [ ]:
import numpy as np

# 1-2-3. Regrouper par destination : nombre de vols, nombre de distances différentes,
#        distance moyenne et retard moyen à l'arrivée
par_dest = (vols.groupby("dest")
                .agg(nb_vols=("flight", "size"),
                     nb_distances=("distance", "nunique"),
                     dist_moy=("distance", "mean"),
                     delay_moy=("arr_delay", "mean"))
                .reset_index())

print("Destinations avec plusieurs distances différentes :", (par_dest["nb_distances"] > 1).sum())
print(par_dest[par_dest["nb_distances"] > 1].head())

# 4. HNL est presque 2 fois plus loin que la 2e destination
print(par_dest.sort_values("dist_moy", ascending=False).head(3))

# 5. On exclut HNL (outlier) et les destinations avec trop peu de vols
sans_hnl = par_dest[(par_dest["dest"] != "HNL") & (par_dest["nb_vols"] > 20)]
print("Corrélation distance / retard :", sans_hnl["dist_moy"].corr(sans_hnl["delay_moy"]).round(2))

# Graphique : un point par destination (taille = nombre de vols) + courbe de tendance
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(sans_hnl["dist_moy"], sans_hnl["delay_moy"],
           s=sans_hnl["nb_vols"] / 50, alpha=0.5, color="steelblue")
coef = np.polyfit(sans_hnl["dist_moy"], sans_hnl["delay_moy"], 2)
x = np.linspace(sans_hnl["dist_moy"].min(), sans_hnl["dist_moy"].max(), 100)
ax.plot(x, np.polyval(coef, x), color="red", linewidth=2, label="Tendance")
ax.set_xlabel("Distance moyenne (miles)")
ax.set_ylabel("Retard moyen à l'arrivée (min)")
ax.set_title("Relation entre la distance et le retard moyen par destination (sans HNL)")
ax.legend()
plt.tight_layout()
plt.savefig("distance_retard.png", dpi=150)
plt.show()

## 3.3 (Q6 à Q9), 3.4 Annulations et données manquantes, 1.1 Durées, 1.2 Carte
Cette cellule recharge les données depuis la base.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# =====================================================================
# PRÉPARATION : recharger les données depuis Supabase
# =====================================================================
vols = pd.read_sql("SELECT * FROM flights", engine)
airports = pd.read_sql("SELECT * FROM airports", engine)
noms = airports.set_index("faa")["name"]

# Dates : départ prévu, départ réel, arrivée prévue, arrivée réelle
vols["sched_dep"] = pd.to_datetime(vols[["year", "month", "day", "hour", "minute"]])
vols["dep"] = vols["sched_dep"] + pd.to_timedelta(vols["dep_delay"], unit="min")
date = pd.to_datetime(vols[["year", "month", "day"]])
vols["sched_arr"] = date + pd.to_timedelta(vols["sched_arr_time"] // 100 * 60 + vols["sched_arr_time"] % 100, unit="min")
vols.loc[vols["sched_arr"] < vols["sched_dep"], "sched_arr"] += pd.Timedelta(days=1)
vols["arr"] = vols["sched_arr"] + pd.to_timedelta(vols["arr_delay"], unit="min")


# =====================================================================
# 3.3 – QUESTIONS 6 À 9
# =====================================================================
print("=" * 70, "\n3.3 – Q6 : destinations les plus touchées et compagnies concernées\n", "=" * 70)
dest = vols.groupby("dest").agg(nb_vols=("flight", "size"), retard_moy=("arr_delay", "mean")).reset_index()
top5 = dest[dest["nb_vols"] >= 100].sort_values("retard_moy", ascending=False).head(5)
top5["nom"] = top5["dest"].map(noms)
print(top5.round(1))
print(vols[vols["dest"].isin(top5["dest"])]
      .groupby(["dest", "carrier"]).agg(nb_vols=("flight", "size"), retard_moy=("arr_delay", "mean")).round(1))

print("\n3.3 – Q7 : résumé statistique")
print(vols[["dep_delay", "arr_delay"]].describe().round(1))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(vols["arr_delay"].dropna(), bins=100, range=(-60, 180), color="steelblue")
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title("Distribution du retard à l'arrivée")
axes[0].set_xlabel("Minutes (limité à -60 / +180)")
axes[0].set_ylabel("Nombre de vols")
axes[1].boxplot([vols["dep_delay"].dropna(), vols["arr_delay"].dropna()], showfliers=False)
axes[1].set_xticks([1, 2], ["Retard au départ", "Retard à l'arrivée"])
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title("Boîtes à moustaches (sans les valeurs extrêmes)")
axes[1].set_ylabel("Minutes")
plt.tight_layout()
plt.savefig("distribution_retards.png", dpi=150)
plt.show()

print("\n3.3 – Q8 : retard moyen par aéroport d'origine")
print(vols.groupby("origin")[["dep_delay", "arr_delay"]].mean().round(1))

print("\n3.3 – Q9 : heure de départ et retard")
par_heure = vols.groupby("hour")["dep_delay"].mean()
print("Corrélation heure / retard :", round(vols["hour"].corr(vols["dep_delay"]), 2))
plt.figure(figsize=(10, 4))
plt.plot(par_heure.index, par_heure.values, marker="o", color="steelblue")
plt.title("Retard moyen au départ selon l'heure prévue")
plt.xlabel("Heure de départ prévue")
plt.ylabel("Retard moyen (min)")
plt.xticks(range(5, 24))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("retard_par_heure.png", dpi=150)
plt.show()


# =====================================================================
# 3.4 – VOLS ANNULÉS ET DONNÉES MANQUANTES
# =====================================================================
print("\n" + "=" * 70, "\n3.4 – Q1 : proportion de données manquantes par colonne\n", "=" * 70)
colonnes_origine = ["year", "month", "day", "dep_time", "sched_dep_time", "dep_delay", "arr_time",
                    "sched_arr_time", "arr_delay", "carrier", "flight", "tailnum", "origin", "dest",
                    "air_time", "distance", "hour", "minute", "time_hour"]
manquants = (vols[colonnes_origine].isna().mean() * 100).round(2)
print(manquants[manquants > 0].sort_values(ascending=False))

print("\n3.4 – Q2 : vols annulés (dep_time ET arr_time manquants)")
vols["annule"] = vols["dep_time"].isna() & vols["arr_time"].isna()
print("Vols annulés :", vols["annule"].sum(), f"({vols['annule'].mean() * 100:.1f} %)")

annul_dest = vols.groupby("dest")["annule"].agg(nb_annules="sum", taux="mean")
annul_dest = annul_dest[annul_dest["nb_annules"] > 0]
annul_dest["taux"] = (annul_dest["taux"] * 100).round(1)
print("\nPar destination (les 10 plus touchées) :")
print(annul_dest.sort_values("nb_annules", ascending=False).head(10))

annul_comp = vols.groupby("carrier")["annule"].agg(nb_annules="sum", taux="mean")
annul_comp["taux"] = (annul_comp["taux"] * 100).round(1)
print("\nPar compagnie :")
print(annul_comp.sort_values("taux", ascending=False))

# Viz : annulations dans le temps (par jour) et par compagnie
vols["date"] = vols["sched_dep"].dt.normalize() - pd.DateOffset(years=8)    # vraie année : 2013
annul_jour = vols.groupby("date")["annule"].sum().asfreq("D")
fig, axes = plt.subplots(2, 1, figsize=(12, 9))
axes[0].plot(annul_jour.index, annul_jour.values, color="firebrick", linewidth=1)
axes[0].set_title("Nombre de vols annulés par jour")
axes[0].set_ylabel("Vols annulés")
t = annul_comp.sort_values("taux")
axes[1].barh(t.index, t["taux"], color="firebrick")
axes[1].set_title("Taux d'annulation par compagnie (%)")
axes[1].set_xlabel("% de vols annulés")
plt.tight_layout()
plt.savefig("annulations.png", dpi=150)
plt.show()
print("Jours avec le plus d'annulations :")
print(vols.groupby("date")["annule"].sum().sort_values(ascending=False).head(5))

print("\n3.4 – Q3 : tri par dep_delay décroissant, NA en premier")
print(vols.sort_values("dep_delay", ascending=False, na_position="first")
          [["sched_dep", "carrier", "flight", "dep_delay"]].head(5))

print("\n3.4 – Q4 : tri par dep_delay décroissant puis dep_time croissant, NA en premier")
print(vols.sort_values(["dep_delay", "dep_time"], ascending=[False, True], na_position="first")
          [["sched_dep", "carrier", "flight", "dep_time", "dep_delay"]].head(5))


# =====================================================================
# 1.1 – CALCUL DE DURÉES
# =====================================================================
print("\n" + "=" * 70, "\n1.1 – Heures converties en minutes depuis minuit\n", "=" * 70)
vols["dep_min"] = vols["dep_time"] // 100 * 60 + vols["dep_time"] % 100
vols["sched_dep_min"] = vols["sched_dep_time"] // 100 * 60 + vols["sched_dep_time"] % 100
print(vols[["dep_time", "dep_min", "sched_dep_time", "sched_dep_min"]].head())

# Relation attendue : dep_delay = dep_min - sched_dep_min (à 24 h près pour les vols partis après minuit)
ecart = (vols["dep_min"] - vols["sched_dep_min"] - vols["dep_delay"]) % 1440
print("Vols où dep_delay = dep_min - sched_dep_min :", f"{(ecart[vols['dep_delay'].notna()] == 0).mean() * 100:.1f} %")

print("\nTop 10 des vols les plus retardés (classement avec ex aequo, comme min_rank) :")
vols["rang"] = vols["dep_delay"].rank(method="min", ascending=False)
print(vols[vols["rang"] <= 10].sort_values("rang")[["rang", "carrier", "flight", "origin", "dest", "dep_delay"]])

# air_time comparé à arr_time - dep_time
vols["duree_brute"] = (vols["arr"] - vols["dep"]).dt.total_seconds() / 60
print("\nÉcart moyen entre (arr_time - dep_time) et air_time :",
      round((vols["duree_brute"] - vols["air_time"]).mean(), 1), "minutes")

# Correction des fuseaux horaires : on ramène tout à l'heure de New York (tz = -5)
tz = airports.set_index("faa")["tz"]
vols["decalage"] = (vols["dest"].map(tz) - (-5)) * 60
vols["duree_corrigee"] = vols["duree_brute"] - vols["decalage"]
print("Écart moyen après correction des fuseaux :",
      round((vols["duree_corrigee"] - vols["air_time"]).mean(), 1), "minutes  (= temps de roulage au sol)")


# =====================================================================
# 1.2 – CARTE DES LIGNES AÉRIENNES
# =====================================================================
coord = airports.set_index("faa")[["lon", "lat"]]
lignes = vols.groupby(["origin", "dest"]).size().rename("nb_vols").reset_index()
lignes["origin_lon"] = lignes["origin"].map(coord["lon"])
lignes["origin_lat"] = lignes["origin"].map(coord["lat"])
lignes["dest_lon"] = lignes["dest"].map(coord["lon"])
lignes["dest_lat"] = lignes["dest"].map(coord["lat"])
lignes = lignes.dropna()                     # les 4 aéroports ajoutés n'ont pas de coordonnées

top100 = lignes.sort_values("nb_vols", ascending=False).head(100)
top5_dest = vols["dest"].value_counts().head(5).index
print("\n1.2 – Les 5 destinations les plus desservies :", list(top5_dest))

fig = go.Figure()
for _, l in top100.iterrows():
    couleur = "firebrick" if l["dest"] in top5_dest else "steelblue"
    fig.add_trace(go.Scattergeo(
        lon=[l["origin_lon"], l["dest_lon"]], lat=[l["origin_lat"], l["dest_lat"]],
        mode="lines", line=dict(width=1 + l["nb_vols"] / 2000, color=couleur),
        opacity=0.6, hoverinfo="text",
        text=f"{l['origin']} → {l['dest']} : {l['nb_vols']} vols", showlegend=False))
dests = top100.drop_duplicates("dest")
fig.add_trace(go.Scattergeo(lon=dests["dest_lon"], lat=dests["dest_lat"], mode="markers",
                            marker=dict(size=4, color="black"), text=dests["dest"], showlegend=False))
fig.update_layout(title="Les 100 lignes aériennes les plus empruntées au départ de New York "
                        "(rouge = vers les 5 destinations principales)",
                  geo=dict(scope="usa", showland=True, landcolor="#f2f2f2"),
                  height=600, margin=dict(l=0, r=0, t=50, b=0))
fig.show()

# MISSION 3 : Application web (Streamlit)
Lancer les cellules dans l'ordre. Le lien `trycloudflare.com` ne fonctionne que tant que Colab tourne.

In [ ]:
!pip install streamlit plotly scikit-learn sqlalchemy psycopg2-binary

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os, json, subprocess, time
from getpass import getpass

mot_de_passe = getpass("Mot de passe Supabase : ")

contenu = f"""[supabase]
user = "postgres.fozjmzskpszdpjeriywn"
password = {json.dumps(mot_de_passe)}
host = "aws-0-eu-west-1.pooler.supabase.com"
port = 5432
database = "postgres"
sslmode = "require"
"""

# On enregistre le fichier aux 2 endroits où Streamlit le cherche
for dossier in ["/root/.streamlit", "/content/.streamlit"]:
    os.makedirs(dossier, exist_ok=True)
    with open(f"{dossier}/secrets.toml", "w") as f:
        f.write(contenu)
print("✅ Secrets enregistrés")

# On arrête l'ancienne application et on la relance
!pkill -f "streamlit run"
time.sleep(2)
subprocess.Popen(["streamlit", "run", "/content/app.py", "--server.port", "8501", "--server.headless", "true"])
print("✅ Application relancée")

In [ ]:
import subprocess, re

# L'application est déjà lancée par la cellule précédente

# 2. Télécharger l'outil qui crée un lien public
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

# 3. Créer le lien
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                          stderr=subprocess.PIPE, text=True)
for ligne in tunnel.stderr:
    lien = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", ligne)
    if lien:
        print(" Ton application :", lien.group())
        break